In [ ]:
# -*- coding: utf-8 -*-
"""
NED — Named Entity Detection (Ejercicio interactivo)
=====================================================
Curso de NLP · Ejercicio práctico sobre extracción de entidades nombradas.

Objetivo:
  Comparar tres enfoques de Named Entity Recognition (NER):
    1. Extracción manual por el estudiante
    2. Extracción automática con un LLM (GPT-4o-mini)
    3. Extracción automática con un modelo NER dedicado (spaCy)

Corpus:
  Oraciones en español basadas en la película "Coco" (Pixar, 2017).

Categorías de entidades permitidas:
  - PERSON : Personas o personajes  (ej. Miguel, Héctor)
  - LOC    : Lugares                (ej. Santa Cecilia, Tierra de los Muertos)
  - ORG    : Organizaciones         (ej. Rivera — como familia/negocio)
  - MISC   : Miscelánea             (ej. Día de Muertos, Recuérdame)

Instrucciones generales:
  1. Ejecuta este script en un entorno con acceso a internet (Colab recomendado).
  2. Necesitas una API key de OpenAI (se te pedirá al inicio si no está configurada).
  3. El script te presentará UN texto del corpus y te guiará paso a paso.
"""

# ── Instalación de dependencias (solo necesario en Colab) ────────────────────
# !pip install spacy openai
# !python -m spacy download es_core_news_lg

import re
import json
import os
import spacy
from getpass import getpass as get_pass
from openai import OpenAI


def parse_json_response(text):
    """
    Extrae y parsea JSON de la respuesta de un LLM,
    eliminando markdown fences (```json ... ```) si están presentes.
    """
    text = text.strip()
    # Quitar bloques de código markdown
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    return json.loads(text)

# ── Configuración de OpenAI ──────────────────────────────────────────────────
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = get_pass("Introduce tu OPENAI_API_KEY: ")
client = OpenAI()

# ── Categorías válidas ───────────────────────────────────────────────────────
ALLOWED = {"PERSON", "LOC", "ORG", "MISC"}

# ── Corpus: oraciones basadas en la película "Coco" ─────────────────────────
textos = [
    "En Santa Cecilia, Miguel admira a Ernesto de la Cruz en un póster del festival y sueña con tocar en la plaza.",
    "En la casa de los Rivera, Abuelita Elena insiste en que la música está prohibida, aunque Miguel practica la guitarra a escondidas.",
    "Durante el Día de Muertos, Miguel entra al mausoleo de Ernesto de la Cruz y toma su guitarra para el concurso.",
    "Al llegar a la Tierra de los Muertos, Miguel conoce a Héctor y a Dante cerca del puente de cempasúchil.",
    "Héctor busca su foto en la ofrenda para poder cruzar y ver a su familia en el mundo de los vivos.",
    "Miguel interpreta Recuérdame para Coco y ella recuerda a su papá.",
    "Imelda enfrenta a Ernesto de la Cruz en el escenario durante su gran fiesta.",
    "En el taller de los Rivera, la familia prepara la ofrenda con la foto de Mamá Imelda.",
]

# ── Gold labels por texto (ground truth) ─────────────────────────────────────
# Cada entrada mapea mención → tipo de entidad.
# NOTA: Se definen aparte del ejercicio para que el estudiante no las vea
#       antes de intentar la extracción por su cuenta.
labels_by_text = {
    0: {"santa cecilia": "LOC", "miguel": "PERSON", "ernesto de la cruz": "PERSON"},
    1: {"rivera": "ORG", "abuelita elena": "PERSON", "miguel": "PERSON"},
    2: {"día de muertos": "MISC", "miguel": "PERSON", "ernesto de la cruz": "PERSON"},
    3: {"tierra de los muertos": "LOC", "miguel": "PERSON", "héctor": "PERSON", "dante": "MISC", "puente de cempasúchil": "LOC"},
    4: {"héctor": "PERSON"},
    5: {"miguel": "PERSON", "recuérdame": "MISC", "coco": "PERSON"},
    6: {"imelda": "PERSON", "ernesto de la cruz": "PERSON"},
    7: {"rivera": "ORG", "mamá imelda": "PERSON"},
}


# ── Función de evaluación ────────────────────────────────────────────────────
def calcular_metricas(gold, pred, nombre):
    """
    Compara un conjunto de predicciones (pred) contra el gold standard (gold).
    Imprime True Positives, False Positives y False Negatives.
    """
    tp = gold & pred
    fp = pred - gold
    fn = gold - pred

    precision = len(tp) / len(pred) if pred else 0.0
    recall = len(tp) / len(gold) if gold else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    print(f"\n{'='*60}")
    print(f"  Resultados para: {nombre}")
    print(f"{'='*60}")
    print(f"  TP (aciertos)   : {len(tp)}")
    print(f"  FP (sobran)     : {len(fp)}")
    print(f"  FN (faltan)     : {len(fn)}")
    print(f"  Precision       : {precision:.2%}")
    print(f"  Recall          : {recall:.2%}")
    print(f"  F1-Score        : {f1:.2%}")
    print(f"{'─'*60}")
    if tp: print(f"  ✅ Aciertos : {sorted(tp)}")
    if fp: print(f"  ⚠️  Sobran   : {sorted(fp)}")
    if fn: print(f"  ❌ Faltan   : {sorted(fn)}")
    print()


# ── Selección del texto a trabajar ───────────────────────────────────────────
i = 3  # Cambia este índice (0-7) para probar con otro texto del corpus
text = textos[i]
gold_dict = labels_by_text[i]
gold_mentions = {m.lower() for m in gold_dict.keys()}
gold_full = {(m.lower(), t) for m, t in gold_dict.items()}

print("\n" + "█" * 60)
print(f"  NAMED ENTITY DETECTION — Texto #{i}")
print("█" * 60)
print(f"\n  📖 Texto:\n  \"{text}\"\n")
print(f"  Categorías válidas: PERSON | LOC | ORG | MISC")
print("─" * 60)


# ══════════════════════════════════════════════════════════════════════════════
# EJERCICIO 1: Extracción de menciones (sin tipo)
# ══════════════════════════════════════════════════════════════════════════════
print("\n🔹 EJERCICIO 1 — Extraer solo las menciones de entidades")
print("─" * 60)
print("  Instrucciones:")
print("    • Lee el texto de arriba con atención.")
print("    • Identifica todos los nombres propios: personas, lugares,")
print("      organizaciones o conceptos relevantes que consideres entidades.")
print("    • Escríbelos separados por coma.")
print("    • No te preocupes por mayúsculas/minúsculas.")
print()
print("  Ejemplo de respuesta:")
print("    Miguel, Santa Cecilia, Ernesto de la Cruz")
print()

user_input_1 = input("  ✏️  Tu respuesta: ")
user_pred_1 = {m.strip().lower() for m in user_input_1.split(",") if m.strip()}

# — ChatGPT (mismo ejercicio) —
prompt_ej1 = (
    f"Extrae todas las entidades nombradas del siguiente texto. "
    f"Retorna SOLO una lista JSON de strings, sin explicación adicional.\n\n"
    f"Texto: \"{text}\""
)
r1 = client.chat.completions.create(
    model="gpt-5.2-chat-latest",
    messages=[{"role": "user", "content": prompt_ej1}],
    temperature=1,
)
gpt_pred_1 = {m.lower() for m in parse_json_response(r1.choices[0].message.content)}

# — Comparación —
calcular_metricas(gold_mentions, user_pred_1, "👤 TÚ (solo menciones)")
calcular_metricas(gold_mentions, gpt_pred_1, "🤖 GPT-4o-mini (solo menciones)")


# ══════════════════════════════════════════════════════════════════════════════
# EJERCICIO 2: Extracción de menciones CON tipo de entidad
# ══════════════════════════════════════════════════════════════════════════════
print("\n🔹 EJERCICIO 2 — Extraer menciones + asignar tipo de entidad")
print("─" * 60)
print("  Instrucciones:")
print("    • Ahora, además de identificar las entidades, clasifícalas.")
print("    • Usa el formato  Mención:TIPO  separando cada una con coma.")
print("    • Tipos válidos: PERSON, LOC, ORG, MISC")
print()
print("  Ejemplo de respuesta:")
print("    Miguel:PERSON, Santa Cecilia:LOC, Día de Muertos:MISC")
print()

user_input_2 = input("  ✏️  Tu respuesta: ")
user_pred_2 = set()
for item in user_input_2.split(","):
    if ":" in item:
        parts = item.rsplit(":", 1)  # rsplit para manejar "Ernesto de la Cruz:PERSON"
        m, t = parts[0].strip().lower(), parts[1].strip().upper()
        if t in ALLOWED:
            user_pred_2.add((m, t))
        else:
            print(f"  ⚠️  Tipo '{t}' no válido para '{m}'. Se ignora. Usa: {ALLOWED}")
    elif item.strip():
        print(f"  ⚠️  '{item.strip()}' no tiene formato Mención:TIPO. Se ignora.")

# — ChatGPT (mismo ejercicio) —
prompt_ej2 = (
    f"Extrae entidades nombradas, de la película de Coco de Pixar y clasifícalas con estos tipos: PERSON, LOC, ORG, MISC.\n"
    f"Retorna SOLO un JSON con esta estructura (sin texto adicional):\n"
    f'  {{"entities": [{{"mention": "...", "type": "..."}}]}}\n\n'
    f'Texto: "{text}"'
)
r2 = client.chat.completions.create(
    model="gpt-5.2-chat-latest",
    messages=[{"role": "user", "content": prompt_ej2}],
    temperature=1,
)
gpt_raw_2 = parse_json_response(r2.choices[0].message.content)["entities"]
gpt_pred_2 = {(e["mention"].lower(), e["type"].upper()) for e in gpt_raw_2}

# — Comparación —
calcular_metricas(gold_full, user_pred_2, "👤 TÚ (mención + tipo)")
calcular_metricas(gold_full, gpt_pred_2, "🤖 GPT-4o-mini (mención + tipo)")


# ══════════════════════════════════════════════════════════════════════════════
# EJERCICIO 3: NER automático con spaCy
# ══════════════════════════════════════════════════════════════════════════════
print("\n🔹 EJERCICIO 3 — NER automático con spaCy (es_core_news_lg)")
print("─" * 60)
print("  Instrucciones:")
print("    • Este paso es automático: spaCy procesará el mismo texto.")
print("    • Observa qué entidades detecta y cómo las clasifica.")
print("    • Compara los resultados con los tuyos y los de ChatGPT.")
print()

try:
    nlp = spacy.load("es_core_news_lg")
except OSError:
    print("  ⏳ Modelo 'es_core_news_lg' no encontrado. Instalando...")
    import subprocess
    subprocess.check_call(["python", "-m", "spacy", "download", "es_core_news_lg"])
    nlp = spacy.load("es_core_news_lg")
doc = nlp(text)

# Mapeo de etiquetas spaCy → las del ejercicio
SPACY2TOY = {"PER": "PERSON", "LOC": "LOC", "ORG": "ORG", "MISC": "MISC"}
spacy_pred = {(e.text.lower(), SPACY2TOY.get(e.label_, "MISC")) for e in doc.ents}

print("  Entidades detectadas por spaCy:")
if doc.ents:
    for e in doc.ents:
        mapped = SPACY2TOY.get(e.label_, "MISC")
        print(f"    • \"{e.text}\" → {e.label_} (mapeado a {mapped})")
else:
    print("    (ninguna entidad detectada)")
print()

calcular_metricas(gold_full, spacy_pred, "🔬 spaCy (mención + tipo)")


# ══════════════════════════════════════════════════════════════════════════════
# RESUMEN FINAL
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "█" * 60)
print("  RESUMEN — ¿Quién lo hizo mejor?")
print("█" * 60)
print()
print("  Reflexiona sobre estas preguntas:")
print("    1. ¿Qué entidades te costó más identificar y por qué?")
print("    2. ¿En qué casos ChatGPT fue mejor o peor que tú?")
print("    3. ¿Qué limitaciones observaste en spaCy para este dominio?")
print("    4. ¿Cómo afecta el contexto (película Coco) a la dificultad?")
print()


████████████████████████████████████████████████████████████
  NAMED ENTITY DETECTION — Texto #3
████████████████████████████████████████████████████████████

  📖 Texto:
  "Al llegar a la Tierra de los Muertos, Miguel conoce a Héctor y a Dante cerca del puente de cempasúchil."

  Categorías válidas: PERSON | LOC | ORG | MISC
────────────────────────────────────────────────────────────

🔹 EJERCICIO 1 — Extraer solo las menciones de entidades
────────────────────────────────────────────────────────────
  Instrucciones:
    • Lee el texto de arriba con atención.
    • Identifica todos los nombres propios: personas, lugares,
      organizaciones o conceptos relevantes que consideres entidades.
    • Escríbelos separados por coma.
    • No te preocupes por mayúsculas/minúsculas.

  Ejemplo de respuesta:
    Miguel, Santa Cecilia, Ernesto de la Cruz

  ✏️  Tu respuesta: Miguel

  Resultados para: 👤 TÚ (solo menciones)
  TP (aciertos)   : 1
  FP (sobran)     : 0
  FN (faltan)     : 4
  Preci